In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns
print("- Libraries imported successfully.")

- Libraries imported successfully.


In [2]:
# Reproducibility

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("-- Using device:", device)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
print("-- Seed set.")


-- Using device: cpu
-- Seed set.


In [3]:
print("-- Defining data transforms...")
img_size = 224

transform_train = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
print("-- Transforms defined.")


-- Defining data transforms...
-- Transforms defined.


In [4]:
print("- Loading dataset...")
data_dir = "D:\\ML-research\\Datasets\\chest_xray_images"  
train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=transform_train)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

class_names = train_dataset.classes
print("--> Dataset loaded. Classes:", class_names)


- Loading dataset...
--> Dataset loaded. Classes: ['bacterial', 'normal', 'viral']


In [5]:
class ResFormer(nn.Module):
    def __init__(self, num_classes):
        super(ResFormer, self).__init__()
        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Identity()  # Remove final fc layer
        self.transformer = nn.TransformerEncoderLayer(d_model=2048, nhead=8)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.resnet(x)
        x = x.unsqueeze(2)  # Shape: [B, 2048, 1]
        x = self.pool(x).squeeze(2)
        x = self.classifier(x)
        return x

model = ResFormer(num_classes=len(class_names)).to(device)
print("--> Model defined.")


d:\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


--> Model defined.


In [6]:
# Loss, optimizer, and scheduler

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

print("--> Optimizer, loss, and scheduler are set.")

--> Optimizer, loss, and scheduler are set.


In [7]:
# Training function

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=20):
    best_acc = 0
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        correct = 0
        total = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

        train_acc = 100. * correct / total

        # Validation
        model.eval()
        correct_val = 0
        total_val = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = outputs.max(1)
                correct_val += preds.eq(labels).sum().item()
                total_val += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_acc = 100. * correct_val / total_val
        scheduler.step(val_acc)

        print(f"--> Epoch {epoch+1}: Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%, Loss: {train_loss/len(train_loader):.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")
            print(f" <--> Epoch {epoch+1}: New best model saved with Val Acc: {val_acc:.2f}%")

        print("-- Classification Report:")
        print(classification_report(all_labels, all_preds, target_names=class_names))
        print("-- Confusion Matrix:")
        print(confusion_matrix(all_labels, all_preds))
    print(f"--> Training complete. Best Accuracy: {best_acc:.2f}%")

train_model(model, train_loader, val_loader, criterion, optimizer, scheduler)


Epoch 1/20: 100%|██████████| 175/175 [29:39<00:00, 10.17s/it]


--> Epoch 1: Train Acc: 79.07%, Val Acc: 85.46%, Loss: 0.5075
 <--> Epoch 1: New best model saved with Val Acc: 85.46%
-- Classification Report:
              precision    recall  f1-score   support

   bacterial       0.78      0.91      0.84       278
      normal       0.92      0.97      0.95       259
       viral       0.88      0.68      0.77       261

    accuracy                           0.85       798
   macro avg       0.86      0.85      0.85       798
weighted avg       0.86      0.85      0.85       798

-- Confusion Matrix:
[[252   7  19]
 [  2 252   5]
 [ 69  14 178]]


Epoch 2/20: 100%|██████████| 175/175 [27:07<00:00,  9.30s/it]


--> Epoch 2: Train Acc: 83.24%, Val Acc: 83.71%, Loss: 0.4029
-- Classification Report:
              precision    recall  f1-score   support

   bacterial       0.74      0.94      0.83       278
      normal       0.91      0.98      0.94       259
       viral       0.92      0.59      0.72       261

    accuracy                           0.84       798
   macro avg       0.86      0.84      0.83       798
weighted avg       0.85      0.84      0.83       798

-- Confusion Matrix:
[[261   5  12]
 [  4 253   2]
 [ 87  20 154]]


Epoch 3/20: 100%|██████████| 175/175 [26:22<00:00,  9.05s/it]


...

--> Epoch 18: Train Acc: 96.40%, Val Acc: 89.10%, Loss: 0.0982
-- Classification Report:
              precision    recall  f1-score   support

   bacterial       0.91      0.81      0.86       278
      normal       0.96      0.99      0.98       259
       viral       0.81      0.88      0.84       261

    accuracy                           0.89       798
   macro avg       0.89      0.89      0.89       798
weighted avg       0.89      0.89      0.89       798

-- Confusion Matrix:
[[225   0  53]
 [  0 257   2]
 [ 21  11 229]]


Epoch 19/20: 100%|██████████| 175/175 [24:26<00:00,  8.38s/it]


--> Epoch 19: Train Acc: 96.52%, Val Acc: 89.97%, Loss: 0.0952
 <--> Epoch 19: New best model saved with Val Acc: 89.97%
-- Classification Report:
              precision    recall  f1-score   support

   bacterial       0.89      0.84      0.87       278
      normal       0.98      0.99      0.98       259
       viral       0.83      0.87      0.85       261

    accuracy                           0.90       798
   macro avg       0.90      0.90      0.90       798
weighted avg       0.90      0.90      0.90       798

-- Confusion Matrix:
[[234   0  44]
 [  0 256   3]
 [ 28   5 228]]


Epoch 20/20: 100%|██████████| 175/175 [24:41<00:00,  8.47s/it]


--> Epoch 20: Train Acc: 96.68%, Val Acc: 88.85%, Loss: 0.0879
-- Classification Report:
              precision    recall  f1-score   support

   bacterial       0.87      0.83      0.85       278
      normal       0.98      0.99      0.99       259
       viral       0.82      0.85      0.83       261

    accuracy                           0.89       798
   macro avg       0.89      0.89      0.89       798
weighted avg       0.89      0.89      0.89       798

-- Confusion Matrix:
[[231   0  47]
 [  1 256   2]
 [ 35   4 222]]
--> Training complete. Best Accuracy: 89.97%
